### AI-Generated

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import sys
import time
import os
sys.path.append("../../../")
import src.experiments.textgenutils as gutils
from src.models.gpt import GPT, GPT2BPETokenizer

/home/wathna/anaconda3/envs/oneforall/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Trained Shakespeare BPE Model

Loading the model trained with BPE tokenization from `out-shakespeare-bpe/`

In [4]:
# Model configuration (matching the trained model)
block_size = 256
n_layers = 6
n_heads = 6
embed_dim = 384
dropout_p = 0.2
bias = False
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")

Using device: cuda


In [6]:
# Load checkpoint
out_dir = '../../../out'
ckpt_path = os.path.join(out_dir, 'ckpt.pt')

checkpoint = torch.load(ckpt_path, map_location=device)
model_args = checkpoint['model_args']

print(f"Model trained for {checkpoint['iter_num']} iterations")
print(f"Best validation loss: {checkpoint['best_val_loss']:.4f}")
print(f"Model args: {model_args}")

Model trained for 500 iterations
Best validation loss: 14.9661
Model args: {'n_layers': 12, 'n_heads': 12, 'embed_dim': 768, 'block_size': 1024, 'bias': False, 'vocab_size': 50257, 'dropout_p': 0.0}


In [7]:
# Initialize model and load weights
model = GPT(**model_args)
state_dict = checkpoint['model']

# Remove '_orig_mod.' prefix if present (from torch.compile)
unwanted_prefix = '_orig_mod.'
for k, v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

model.load_state_dict(state_dict)
model.eval()
model.to(device)

print("Model loaded successfully!")

Model loaded successfully!


In [8]:
# Initialize tokenizer
tokenizer = GPT2BPETokenizer()
print(f"Tokenizer vocab size: {len(tokenizer.token_to_index)}")

Tokenizer vocab size: 50257


# KV Cache Profiling

Profile generation time and memory consumption with and without KV caching.

In [9]:
def profile_kv_cache_generation_time_and_memory_consumption(prompt: str,
                                                            n_tokens_to_generate: int):
    """Profile generation with and without KV cache."""
    indices = tokenizer.encode(prompt)

    print('Prompt:', prompt)
    print('Tokens to generate:', n_tokens_to_generate)
    model_param_fp32s = sum(p.numel() for p in model.parameters())
    print('Model Param Memory Consumption:', f'{model_param_fp32s / 1e6 * 4:.1f} MB')
    print()

    for use_kv_cache in [False, True]:
        if use_kv_cache:
            print('USING KV CACHE')
            print('==============')
        else:
            print('NOT USING KV CACHE')
            print('==================')

        model.clear_kv_cache()
        start_time = time.time()

        generator = gutils.generate_text(model, tokenizer, prompt,
                                         n_tokens_to_gen=n_tokens_to_generate,
                                         top_k=100,
                                         top_p=0.95,
                                         sample=True,
                                         print_stream=False,
                                         use_kv_cache=use_kv_cache)

        response = ''.join(list(generator))
        end_time = time.time()

        if use_kv_cache:
            kv_cache_fp32s = model.decoder_blocks[0].attn.kv_cache[0].numel()
            peak_kv_cache_fp32s = (len(indices) + n_tokens_to_generate) * kv_cache_fp32s // len(indices)
        else:
            peak_kv_cache_fp32s = 0

        print('Generation time:'.ljust(28), f'{end_time - start_time:.1f} sec')
        print('KV Cache Memory Consumption:'.ljust(28), f'{peak_kv_cache_fp32s / 1e6 * 4:.1f} MB')
        print()

    print('Response:', response.lstrip())

In [10]:
# Test with a Shakespeare-style prompt
profile_kv_cache_generation_time_and_memory_consumption(
    'ROMEO:\nBut soft! What light through yonder window breaks?',
    n_tokens_to_generate=230
)

Prompt: ROMEO:
But soft! What light through yonder window breaks?
Tokens to generate: 230
Model Param Memory Consumption: 497.8 MB

NOT USING KV CACHE
Generation time:             3.1 sec
KV Cache Memory Consumption: 0.0 MB

USING KV CACHE
Generation time:             1.6 sec
KV Cache Memory Consumption: 12.3 MB

Response: MyMyMyMyMyMyMyMyMyMyMyMyMyMyAA:























MyMyMyMyMyMyMyMyMyMyMyMyMyMy
















MyMyMyMyMy Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord Lord:






















MyMyMyMyMy love love loveA:















# Generate Shakespeare-style Text

Now let's generate some Shakespeare-style dialogue using the trained model.

In [27]:
def generate_shakespeare(prompt: str, 
                        n_tokens: int = 200,
                        temperature: float = 0.8,
                        top_k: int = 200,
                        top_p: float = 0.95,
                        use_kv_cache: bool = True):
    """Generate Shakespeare-style text with KV caching."""
    model.clear_kv_cache()
    
    print(f"Prompt:\n{prompt}")
    print("\n" + "="*80)
    print("Generated text:")
    print("="*80)
    
    response = gutils.generate_text(
        model, 
        tokenizer, 
        prompt,
        n_tokens_to_gen=n_tokens,
        top_k=top_k,
        top_p=top_p,
        temperature=temperature,
        sample=True,
        print_stream=True,
        use_kv_cache=use_kv_cache
    )
    
    print("\n" + "="*80)
    return response

In [28]:
# Example 1: Generate from a character name
generate_shakespeare(
    "HENRY V:\n",
    n_tokens=200,
    temperature=0.8
)

Prompt:
HENRY V:


Generated text:
Is yet our country's life
That thou mayst stay the eye of all father's hand.

KING RICHARD II:
Ay, my gracious lord.

NORTHUMBERLAND:

KING RICHARD II:
Yes, my lord?

NORTHUMBERLAND:
Come, sir, in thy eyes makes me by thy birth beheld!

DUKE OF YORK:
No, I'll stand to the king.

HENRY BOLINGBROKE:
Say you be fair?

JOHN OF GAUNT:
What is the man of
To undertake the king; I'll have got the sea and as a king?

DUKE OF YORK:
The next way of Lancaster!
What, is it not thy life that you have been been an oath and to make you to my good lord, you are.

JOHN OF GAUNT:
Let


"Is yet our country's life\nThat thou mayst stay the eye of all father's hand.\n\nKING RICHARD II:\nAy, my gracious lord.\n\nNORTHUMBERLAND:\n\nKING RICHARD II:\nYes, my lord?\n\nNORTHUMBERLAND:\nCome, sir, in thy eyes makes me by thy birth beheld!\n\nDUKE OF YORK:\nNo, I'll stand to the king.\n\nHENRY BOLINGBROKE:\nSay you be fair?\n\nJOHN OF GAUNT:\nWhat is the man of\nTo undertake the king; I'll have got the sea and as a king?\n\nDUKE OF YORK:\nThe next way of Lancaster!\nWhat, is it not thy life that you have been been an oath and to make you to my good lord, you are.\n\nJOHN OF GAUNT:\nLet"

In [29]:
# Example 2: Generate from a famous quote
generate_shakespeare(
    "HAMLET:\nTo be, or not to be,",
    n_tokens=150,
    temperature=0.9
)

Prompt:
HAMLET:
To be, or not to be,

Generated text:
Why do it.

KING RICHARD III:
Good Clarence, I know he that Richmond?

BUCKINGHAM:
Let me will be so doth make a' soldiers;
And over him in his friends.

KING RICHARD III:

NORFOLK:
Tell me speak.

KING RICHARD III:
Good Norfolk,
So will I have been more.

NORFOLK:
To bear the rest will I have heard.

NORFOLK:
O, go with me.

NORFOLK:
A royal sir! he is not that I know not.

NORFOLK:



"Why do it.\n\nKING RICHARD III:\nGood Clarence, I know he that Richmond?\n\nBUCKINGHAM:\nLet me will be so doth make a' soldiers;\nAnd over him in his friends.\n\nKING RICHARD III:\n\nNORFOLK:\nTell me speak.\n\nKING RICHARD III:\nGood Norfolk,\nSo will I have been more.\n\nNORFOLK:\nTo bear the rest will I have heard.\n\nNORFOLK:\nO, go with me.\n\nNORFOLK:\nA royal sir! he is not that I know not.\n\nNORFOLK:\n"

In [31]:
# Example 3: Start a dialogue between multiple characters
generate_shakespeare(
    """ROMEO:
But, soft! what light through yonder window breaks?

JULIET:
""",
    n_tokens=230,
    temperature=0.85
)

Prompt:
ROMEO:
But, soft! what light through yonder window breaks?

JULIET:


Generated text:
Nurse:
Then, sir!

JULIET:

JULIET:
Or she is not know.

JULIET:
Go she will not be pay to be
From your place;
For we'll do beseech you and good night.

Nurse:
Well, I'll keep a good.

ROMEO:
My name is dead!

ROMEO:
O Romeo!

ROMEO:
Madam!

ROMEO:
What says certain of thee?

ROMEO:
Marry, she's to-night!

ROMEO:
O, thou art not.

ROMEO:
Good night!

JULIET:
Ay me, sister!

ROMEO:
Go with me.

JULIET:
Ay, the world-mark!

ROMEO:
Come me a peev?

JULIET:
My Lord of death;
And tell thee, thy mistress!

JULI


"Nurse:\nThen, sir!\n\nJULIET:\n\nJULIET:\nOr she is not know.\n\nJULIET:\nGo she will not be pay to be\nFrom your place;\nFor we'll do beseech you and good night.\n\nNurse:\nWell, I'll keep a good.\n\nROMEO:\nMy name is dead!\n\nROMEO:\nO Romeo!\n\nROMEO:\nMadam!\n\nROMEO:\nWhat says certain of thee?\n\nROMEO:\nMarry, she's to-night!\n\nROMEO:\nO, thou art not.\n\nROMEO:\nGood night!\n\nJULIET:\nAy me, sister!\n\nROMEO:\nGo with me.\n\nJULIET:\nAy, the world-mark!\n\nROMEO:\nCome me a peev?\n\nJULIET:\nMy Lord of death;\nAnd tell thee, thy mistress!\n\nJULI"

# Compare Temperature Settings

Different temperature values affect the creativity vs. coherence tradeoff.

In [32]:
prompt = "MACBETH:\nIs this a dagger which I see before me,"

for temp in [0.5, 0.8, 1.2]:
    print(f"\n{'='*80}")
    print(f"Temperature: {temp}")
    print('='*80)
    generate_shakespeare(prompt, n_tokens=100, temperature=temp)
    print()


Temperature: 0.5
Prompt:
MACBETH:
Is this a dagger which I see before me,

Generated text:
And I will be so;
For I am no man in this night.

BUCKINGHAM:
Then I have been a grace
But I have it.

GLOUCESTER:
Then 'twere no less!

GLOUCESTER:
I will not be avoided;
But, I will not speak.

BUCKINGHAM:
The gracious lord,
And therefore I will not be it to-morrow.


Temperature: 0.8
Prompt:
MACBETH:
Is this a dagger which I see before me,

Generated text:
I cannot say the house.

BUCKINGHAM:
Come, come on, and I know not.

GLOUCESTER:
Good time to the king that I am yours,
Or, if I am sure
That I am no mother of Clarence,
By the city is that I should live
The king's mother,
Would I should not--

GLOUCESTER:
Then would I'll not be gone?




Temperature: 1.2
Prompt:
MACBETH:
Is this a dagger which I see before me,

Generated text:
My pity doth fall of their light as far now now
His son dares he
At any good to your fair flesh
Be record
For 'twas being to the queen:
What time your shame, being l

# Using model.generate_sample() Directly

Alternative way to generate without streaming (simpler but less flexible).

In [33]:
start = "KING LEAR:\nHow sharper than a serpent's tooth it is"
start_ids = tokenizer.encode(start)
x = torch.tensor(start_ids, dtype=torch.long, device=device)[None, ...]

print(f"Prompt: {start}")
print(f"Encoded to {len(start_ids)} tokens\n")
print("="*80)

# Generate
with torch.no_grad():
    y = model.generate_sample(x, max_new_tokens=150, temperature=0.8, top_k=200)

# Decode and print
generated_text = tokenizer.decode(y[0].tolist())
print(generated_text)
print("="*80)

Prompt: KING LEAR:
How sharper than a serpent's tooth it is
Encoded to 14 tokens

KING LEAR:
How sharper than a serpent's tooth it is a poor soul
To hear thee and do not for their grandsire
My Lord Aumerle.

KING RICHARD II:
And I'll swear it go.
Look here comes dead.

CLIFFORD:
Not to die?

EXETER:
Ay, I'll do't.

KING HENRY VI:

CLIFFORD:

QUEEN MARGARET:
Why, we must put it.

QUEEN ELIZABETH:
What authority,
When thou sleep'st me,
Or yours canst not be.

KING RICHARD III:
As thou hast thou not bid me?




# Model Statistics

In [34]:
print(f"Model Architecture:")
print(f"  Layers: {model.n_layers}")
print(f"  Heads: {model.n_heads}")
print(f"  Embedding dimension: {model.embed_dim}")
print(f"  Vocab size: {model.vocab_size}")
print(f"  Block size (context): {model.block_size}")
print()
print(f"Total parameters: {model.get_num_params():,}")
print(f"Total parameters (non-embedding): {model.get_num_params(non_embedding=True):,}")
print()
print(f"Training stats:")
print(f"  Iterations trained: {checkpoint['iter_num']:,}")
print(f"  Best validation loss: {checkpoint['best_val_loss']:.4f}")

Model Architecture:
  Layers: 6
  Heads: 6
  Embedding dimension: 384
  Vocab size: 50257
  Block size (context): 256

Total parameters: 10,750,080
Total parameters (non-embedding): 10,750,080

Training stats:
  Iterations trained: 1,500
  Best validation loss: 5.6641
